# Inyector de Drift — Physionet (Sepsis): covariate + concept drift condicionado

Este notebook documenta y valida el **inyector de drift mejorado** para Physionet, en respuesta al
Gap 2 de `plan/plan_semanal_pfc2.md`.

| Mecanismo | Módulo | Qué cambia | Estado |
|---|---|---|---|
| **Covariate shift** | `darl.drift.DriftInjector` | `P(X)` vía mezcla Beta por variable | Reusado sin cambios (ya validado) |
| **Concept drift estructurado** | `darl.drift.ConceptDriftInjector` (**nuevo**) | `P(Y\|X)` vía amplificación del logit de un modelo de referencia | Nuevo en este notebook |
| **Concept drift — ruido (control negativo)** | `darl.drift.ConceptDriftInjector` (`mechanism="noise_control"`) | `Y` vía flip simétrico aleatorio | Mecanismo anterior, conservado solo como control |

El problema que resuelve: el inyector anterior generaba concept drift con un *label flipping* simétrico
(`p_flip = 0.45 * severity`), que introduce **ruido no estructurado**, no un cambio interpretable en
`P(Y|X)`. Ese mecanismo se conserva aquí únicamente como control negativo explícito.


## Dataset card — mecanismo de concept drift condicionado

**Dataset:** Physionet 2019 Sepsis Challenge (vía TableShift), variable objetivo `SepsisLabel`.

**Variables usadas (`VITALS`):** `HR` (frecuencia cardiaca), `SBP`/`MAP` (presión arterial), `Resp`
(frecuencia respiratoria), `Temp` (temperatura) — son los signos vitales clínicamente asociados al
deterioro por sepsis (taquicardia, taquipnea, hipotensión), y ya son las variables tratadas como
"más significativas" en `physionet_selective_update.ipynb`.

**Mecanismo (`mechanism="logit_shift"`):**

1. Se ajusta una regresión logística de referencia sobre `VITALS` (estandarizadas) → `SepsisLabel`
   en `df_train`. Sus coeficientes `beta_i` definen el logit base `logit0(x)`.
2. Para cada fila del conjunto objetivo se calcula `delta(x) = sum(beta_i * z_i(x))` — la contribución
   de los signos vitales al riesgo ya aprendido por el modelo.
3. El logit drifteado es `logit_new(x) = logit0(x) + severity * delta(x)`: la relación entre signos
   vitales y riesgo de sepsis se **amplifica** progresivamente con la severidad, en vez de invertirse
   o volverse ruido.
4. Cada fila se reetiqueta con probabilidad `p_flip(x) = |sigmoid(logit_new(x)) - sigmoid(logit0(x))|`
   — las filas donde el drift realmente mueve el riesgo son las que tienen más probabilidad de
   recibir una nueva etiqueta muestreada de `sigmoid(logit_new(x))`.

**Por qué es plausible clínicamente:** representa un escenario donde los signos de deterioro
fisiológico (taquicardia, taquipnea, hipotensión) se vuelven **más predictivos** de sepsis con el
tiempo — p. ej. por cambio en el case-mix hacia pacientes más graves, o por un protocolo clínico que
captura antes el deterioro. No es la única forma plausible de concept drift, pero es interpretable y
verificable.

**Garantía por construcción:** en `severity=0`, `logit_new == logit0` en todas las filas, por lo tanto
`p_flip(x) == 0` y las etiquetas quedan exactamente iguales (ver validación más abajo). El mecanismo
**nunca toca `X`** — solo `Y` — así que un detector que observe solo `X` (PSI/KS) debería permanecer en
su rango nulo bajo concept drift puro.

**Limitaciones declaradas:** el modelo de referencia es una regresión logística simple (no captura
interacciones no lineales); el `delta(x)` amplifica *todas* las variables de `VITALS` a la vez, no
subgrupos específicos (a diferencia del mecanismo propuesto para `diabetes_readmission` en el Gap 2);
el resampleo de `Y` es estocástico, por lo que dos corridas con distinta semilla producen conjuntos
de filas reetiquetadas ligeramente distintos aunque la severidad sea la misma.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

from darl.data.get_dataset import load_dataset
from darl.drift import ConceptDriftInjector, DriftInjector
from darl.evaluation import ks_stat, psi_numeric
from darl.visualization.drift_plots import plot_conditional_risk, plot_numeric_drift_grid

SEED = 42
np.random.seed(SEED)

VITALS = ["HR", "SBP", "MAP", "Resp", "Temp"]
LABEL_COL = "SepsisLabel"

# Configuración de covariate drift por variable (misma que physionet_selective_update.ipynb)
NUM_CFG = {
    "HR": "high",
    "SBP": "low",
    "MAP": "low",
    "Resp": "high",
    "Temp": "extreme",
}

print("Imports OK")

ModuleNotFoundError: No module named 'numpy'

## 1. Carga del dataset Physionet

In [ ]:
dset = load_dataset("physionet")

X_train, y_train, _, _ = dset.get_pandas(split="train")
X_target, y_target, _, _ = dset.get_pandas(split="id_test")

df_train = X_train.copy()
df_train[LABEL_COL] = y_train.values

df_target = X_target.copy()
df_target[LABEL_COL] = y_target.values

print(f"Train  : {df_train.shape}")
print(f"Target : {df_target.shape}")
print(f"Prevalencia Sepsis Train : {y_train.mean():.4f}")
print(f"Prevalencia Sepsis Target: {y_target.mean():.4f}")
print(f"VITALS presentes: {[c for c in VITALS if c in df_target.columns]}")

## 2. Covariate shift (reuso de `DriftInjector`, sin cambios)

Este mecanismo ya está validado (ver Gap 1 / `physionet_selective_update.ipynb`): mezcla Beta por
variable, controlada por `drift_severity`. Se reusa tal cual para mantener consistencia entre gaps.

In [ ]:
cov_inj = DriftInjector(random_state=SEED)
cov_inj.fit(df_train, numeric_cols=VITALS)

COV_SEVERITIES = [0.2, 0.5, 0.8]
cov_frames = {}
cov_summaries = []

for sev in COV_SEVERITIES:
    df_cov, meta_cov = cov_inj.transform(
        df_target, drift_severity=sev, drift_type="covariate", numeric_drift_config=NUM_CFG
    )
    cov_frames[sev] = df_cov
    summary = DriftInjector.summary(meta_cov)
    summary["severity"] = sev
    cov_summaries.append(summary)

df_covariate_summary = pd.concat(cov_summaries)
df_covariate_summary

In [ ]:
fig = plot_numeric_drift_grid(df_target, cov_frames[0.5], cols=VITALS, ncols=3, bins=60)
plt.suptitle("Covariate shift (sev=0.5) — VITALS antes vs después", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## 3. Concept drift estructurado (`ConceptDriftInjector`, nuevo)

Se ajusta el modelo de referencia sobre `df_train` y se aplican ambos mecanismos
(`logit_shift` y `noise_control`) sobre `df_target`, a las mismas severidades, para poder
compararlos directamente.

In [ ]:
concept_inj = ConceptDriftInjector(random_state=SEED)
concept_inj.fit(df_train, target_vars=VITALS, label_col=LABEL_COL)

print("Coeficientes del modelo de referencia (signo = dirección clínica esperada):")
for var, coef in concept_inj.coef_.items():
    print(f"  {var:6s}: {coef:+.4f}")
print(f"  intercept: {concept_inj.intercept_:+.4f}")

In [ ]:
CONCEPT_SEVERITIES = [0.0, 0.2, 0.5, 0.8, 1.2]

concept_frames = {"logit_shift": {}, "noise_control": {}}
concept_summaries = []

for mechanism in ["logit_shift", "noise_control"]:
    for sev in CONCEPT_SEVERITIES:
        df_drift, meta = concept_inj.transform(df_target, severity=sev, mechanism=mechanism)
        concept_frames[mechanism][sev] = df_drift
        concept_summaries.append(ConceptDriftInjector.summary(meta))

df_concept_summary = pd.concat(concept_summaries).reset_index()
df_concept_summary

## 4. Validación — `P(X)` no debe moverse bajo concept drift puro

Si el mecanismo funciona como se diseñó, `logit_shift` a cualquier severidad no debe alterar la
distribución de `VITALS` (solo toca `Y`). Se compara contra `df_target` original con KS y PSI —
ambos deberían quedar en su rango nulo (KS≈0, PSI≈0) en todas las severidades.

In [ ]:
stability_rows = []
for sev in CONCEPT_SEVERITIES:
    df_drift = concept_frames["logit_shift"][sev]
    for col in VITALS:
        ks = ks_stat(df_target[col], df_drift[col])
        psi = psi_numeric(df_target[col], df_drift[col])
        stability_rows.append({"severity": sev, "variable": col, "ks_stat": ks["ks_stat"], "psi": psi})

df_stability = pd.DataFrame(stability_rows)
print("Máximo KS observado:", df_stability["ks_stat"].max())
print("Máximo PSI observado:", df_stability["psi"].abs().max())
df_stability.pivot(index="variable", columns="severity", values="ks_stat")

## 5. Gráfica clave — relación condicional `X → Y`, antes vs después

A diferencia de los histogramas de arriba (que muestran cómo se mueve `X`), esta gráfica muestra
cómo se mueve la relación `X → Y`: para cada variable, el riesgo empírico `P(Y=1)` por bins,
antes vs después del drift. Se compara `logit_shift` (estructurado) contra `noise_control` (ruido)
a la misma severidad, para evidenciar que no son el mismo tipo de cambio.

In [ ]:
SEV_PLOT = 0.8

fig1 = plot_conditional_risk(
    df_target, df_target[LABEL_COL],
    concept_frames["logit_shift"][SEV_PLOT], concept_frames["logit_shift"][SEV_PLOT][LABEL_COL],
    cols=VITALS, ncols=3, after_label=f"logit_shift (sev={SEV_PLOT})",
)
plt.show()

fig2 = plot_conditional_risk(
    df_target, df_target[LABEL_COL],
    concept_frames["noise_control"][SEV_PLOT], concept_frames["noise_control"][SEV_PLOT][LABEL_COL],
    cols=VITALS, ncols=3, after_label=f"noise_control (sev={SEV_PLOT})",
)
plt.show()

## 6. Validación — degradación de AUC ordenada por severidad

Se evalúa el propio modelo de referencia (`concept_inj.predict_proba`) sobre los datos drifteados:
si el mecanismo estructurado realmente mueve `P(Y|X)`, el AUC del modelo congelado debería degradarse
de forma ordenada con la severidad. Se compara contra `noise_control` para contraste.

In [ ]:
auc_rows = []
for mechanism in ["logit_shift", "noise_control"]:
    for sev in CONCEPT_SEVERITIES:
        df_drift = concept_frames[mechanism][sev]
        y_score = concept_inj.predict_proba(df_drift)
        auc = roc_auc_score(df_drift[LABEL_COL], y_score)
        auc_rows.append({"mechanism": mechanism, "severity": sev, "auc_reference_model": auc})

df_auc = pd.DataFrame(auc_rows)
df_auc_pivot = df_auc.pivot(index="severity", columns="mechanism", values="auc_reference_model")
df_auc_pivot

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
for mechanism, color in [("logit_shift", "#d62728"), ("noise_control", "#7f7f7f")]:
    sub = df_auc[df_auc["mechanism"] == mechanism]
    ax.plot(sub["severity"], sub["auc_reference_model"], marker="o", label=mechanism, color=color)
ax.set_xlabel("Severidad")
ax.set_ylabel("AUC del modelo de referencia")
ax.set_title("Degradación de AUC por severidad — logit_shift vs noise_control", fontweight="bold")
ax.legend()
ax.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

## 7. Guardar artefactos

In [ ]:
from darl.utils import find_project_root

OUT_DIR = find_project_root() / "outputs" / "metrics"
OUT_DIR.mkdir(parents=True, exist_ok=True)

df_covariate_summary.to_csv(OUT_DIR / "gap2_covariate_drift_physionet.csv")
df_concept_summary.to_csv(OUT_DIR / "gap2_concept_drift_physionet.csv", index=False)
df_stability.to_csv(OUT_DIR / "gap2_concept_x_stability_physionet.csv", index=False)
df_auc.to_csv(OUT_DIR / "gap2_concept_auc_by_severity_physionet.csv", index=False)

print("Artefactos guardados en", OUT_DIR)

## 8. Conclusiones

- El mecanismo `logit_shift` cumple la garantía de diseño: `severity=0` dejó las etiquetas
  intactas y `P(X)` permaneció estable (KS/PSI ≈ 0) en todas las severidades — el drift es
  puramente conceptual, no covariate encubierto.
- La relación `X → Y` se hace visiblemente más pronunciada con la severidad (sección 5), y el AUC
  del modelo congelado se degrada de forma ordenada (sección 6) — a diferencia de `noise_control`,
  que degrada por ruido no interpretable.
- `noise_control` queda documentado y disponible solo como control negativo, tal como pide el Gap 2
  del plan semanal — no se usa como mecanismo principal de ningún experimento futuro.

**Próximos pasos:** replicar este mismo patrón (mecanismo condicionado + dataset card + validación
P(X)/AUC) para `diabetes_readmission` con un mecanismo por subgrupos (no por logit-shift continuo,
ya que ahí el Gap 2 pide condicionar por fuente de admisión / historial / utilización de urgencias);
y conectar `ConceptDriftInjector` con los experimentos de compatibilidad Stage1↔Stage2 (Gap 1) para
evaluar A1–A4 bajo este concept drift más realista, en vez del label-flip anterior.